In [53]:
# import all libraries
import pandas as pd
import numpy as np
import re
import time
import json
from typing import Dict
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
# from autocorrect import Speller
from spellchecker import SpellChecker

# Loading and Preprocessing

In [54]:
# import and read dataset
df = pd.read_csv('../datasets/Fin_lab-PRProject_dataset.csv')
df.head()

,Unnamed: 0,recommendationid,language,review,Reaction
0,0,77057085,english,Is good. Do play.,0
1,1,77052689,english,AAAAAAAA,0
2,2,77049252,english,Fun game,1
3,3,77049089,english,"Great game, worth every penny!",0
4,4,35101272,english,Like,0


In [55]:
# get only 5000 random row samples
sample_df = df.sample(n=5000, random_state=42)
sample_df.head()

,Unnamed: 0,recommendationid,language,review,Reaction
30583,83,27076016,english,This game has got to be one of the most memora...,0
28537,37,28045907,english,The weirdest shit I've ever played and yet ins...,1
11192,92,21598005,english,This game + the original Binding of Isaac got ...,1
18237,37,16024475,english,Binding of Isaac: Cumtopia (REMASTERED),0
16235,35,52680381,english,great,1


In [56]:
# check the dataset null/nan values
sample_df[sample_df['review'].isnull()]

,Unnamed: 0,recommendationid,language,review,Reaction
6178,78,65829672,english,NaN,1
44321,21,12940658,english,NaN,1
11879,79,58009191,english,NaN,0
10575,75,59775117,english,NaN,0
10907,7,59373379,english,NaN,1
41587,87,14038822,english,NaN,0
1331,31,74465830,english,NaN,1
2915,15,71696443,english,NaN,0


In [57]:
# check types in 'review' column
sample_df['review'].apply(type).value_counts()

review
<class 'str'>      4992
<class 'float'>       8
Name: count, dtype: int64

In [58]:
# contraction mapping
CONTRACTION_MAP = {
    "ain't": "is not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "i've" : "i have",
    "i'd" : "i had",
    "you've" : "you have",
    "you'd" : "you had",
    "we've" : "we have",
    "we'd" : "we had",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "wasn't" : "was not",
    "that's": "that is",
    "you'll" : "you will"
}

In [59]:
# MODULE DECLARATIONS
spellChecker = SpellChecker(language='en')
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# HELPER FUNCTIONS

# tokenize text
def tokenize_text(text):
    word_tokens = word_tokenize(text)
    return word_tokens

# clear rows that has null values
def clear_null_values(text):
    if not isinstance(text, str):
        return ""
    return text

# remove punctuations
def remove_punctuations(text):
    punctuations = '''!()-[]{};:'.'"\,<>./?@#$%^–—&_0123456789~*+'''
    # remove punctuations from the text
    no_punct = ""
    for char in text:
        if char not in punctuations:
            no_punct = no_punct + char
    return no_punct

# remove repeating characters
def remove_repeating_characters(tokens):
    repeatedPattern = re.compile(r'(\w*)(\w)\2(\w*)')
    matchSubstitution = r'\1\2\3'
    def replace(oldWord):
        if wordnet.synsets(oldWord):
            return oldWord
        newWord = repeatedPattern.sub(matchSubstitution, oldWord)
        return replace(newWord) if newWord != oldWord else newWord
    correctTokens = [replace(word) for word in tokens]
    return correctTokens

# remove contractions
def expand_contractions(text, contraction_mapping=CONTRACTION_MAP):

    contractions_pattern = re.compile('({})'.format('|'.join(contraction_mapping.keys())), flags=re.IGNORECASE|re.DOTALL)

    def expand_match(contraction):
        match = contraction.group(0)
        first_char = match[0]
        expanded_contraction = contraction_mapping.get(match) if contraction_mapping.get(match) else contraction_mapping.get(match.lower())
        expanded_contraction = first_char + expanded_contraction[1:]
        return expanded_contraction
    
    expanded_text = contractions_pattern.sub(expand_match, text)
    expanded_text = re.sub("'", "", expanded_text)
    return expanded_text

# remove stopwords
myWordDictionary = []
def remove_stopwords_orig(tokens):
    stopWords = set(stopwords.words('english'))
    wordsFiltered = []

    for w in tokens:
        # skip None or empty values
        if not isinstance(w, str) or not w.strip():
            continue
        if w not in stopWords:
            myWordDictionary.append(w)
            wordsFiltered.append(w)
            wordsFiltered.append(' ')  # spacing

    return "".join(wordsFiltered)

def build_correction_map(texts: pd.Series, spell=spellChecker) -> Dict[str, str]:
    print("Building spelling correction map...")
    start_time = time.time()

    all_text = ' '.join(texts.astype(str))
    all_text = re.sub(r'[^a-zA-Z]', ' ', all_text).lower()
    unique_words = set(all_text.split())

    misspelled = spell.unknown(unique_words)
    correction_map = {word: spell.correction(word) for word in misspelled}

    end_time = time.time()
    print(f"Built correction map in {end_time - start_time:.2f}s")
    print(f"Found {len(misspelled)} typos, mapped {len(correction_map)} corrections.")
    return correction_map

def apply_corrections(tokens, correction_map):
    return [correction_map.get(word, word) for word in tokens]

<>:21: SyntaxWarning: invalid escape sequence '\,'
<>:21: SyntaxWarning: invalid escape sequence '\,'
C:\Users\mosqu\AppData\Local\Temp\ipykernel_19368\1610943146.py:21: SyntaxWarning: invalid escape sequence '\,'
  punctuations = '''!()-[]{};:'.'"\,<>./?@#$%^–—&_0123456789~*+'''


In [60]:
# CLEAN TEXT PIPELINE
def clean_text_pipeline(text, correction_map=None, do_spellcheck=True):
    # 1 - handle nulls
    text = clear_null_values(text)

    # 2 - expand contractions
    text = expand_contractions(text)

    # 3 - remove punctuations
    text = remove_punctuations(text)

    # 4 - turn all text to lowercase
    text = text.lower()

    # 5 - tokenize text
    tokens = tokenize_text(text)

    # 6 - remove repeating characters
    tokens = remove_repeating_characters(tokens)

    # 7 - spell check
    if do_spellcheck and correction_map is not None:
        tokens = apply_corrections(tokens, correction_map)

    # clean invalid tokens
    tokens = [t for t in tokens if isinstance(t, str) and t.strip()]

    # 8 - remove stopwords
    tokens = remove_stopwords_orig(tokens)

    # 9 - stemming
    tokens = [stemmer.stem(word) for word in tokens]

    # 10 - lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # 11 - join back the cleaned text into a single string
    final_cleaned_text = ''.join(tokens)

    return final_cleaned_text.strip()


# Cleaning

In [61]:
# build correction map
# correction_map = build_correction_map(sample_df['review'])

In [62]:
# dump correction map as json
# with open('../json_dump/correction_map.json', 'w') as f:
    # json.dump(correction_map, f)

In [63]:
with open('../json_dump/correction_map.json') as f:
    correction_map = json.load(f)

In [64]:
# apply cleaning
sample_df['cleaned_review'] = sample_df['review'].apply(lambda x: clean_text_pipeline(x, correction_map=correction_map, do_spellcheck=True))

In [65]:
sample_df.head()

,Unnamed: 0,recommendationid,language,review,Reaction,cleaned_review
30583,83,27076016,english,This game has got to be one of the most memora...,0,game got one memorable games played truly with...
28537,37,28045907,english,The weirdest shit I've ever played and yet ins...,1,weirdest shit ever played yet insanely fun def...
11192,92,21598005,english,This game + the original Binding of Isaac got ...,1,game original binding isaac got games love gam...
18237,37,16024475,english,Binding of Isaac: Cumtopia (REMASTERED),0,binding isaac utopia remastered
16235,35,52680381,english,great,1,great


# Extraction (BOW and TFIDF)

In [66]:
n_gram_categ = 'unigrams'

# BAG OF WORDS EXTRACTION

# corpus is the collection of text docs
# n of n-gram can be modified
# max feature is limited to 500 to keep top 500 most frequent words, can be changed though
def bow_extractor(corpus, ngram_range=(1,1)):
    vectorizer = CountVectorizer(min_df=1, ngram_range=ngram_range)
    features = vectorizer.fit_transform(corpus)
    return vectorizer, features

# display features
def display_features(features, feature_names):
    df = pd.DataFrame(data=features, columns=feature_names)
    print(df)

# save features to csv
def save_features(features, feature_names, filename):
    df = pd.DataFrame(data=features, columns=feature_names)
    df.to_csv(filename, index=False)
    print(f"✅ Features extracted and saved to {filename}")

# TFIDF EXTRACTION
def tfidf_transformer(bow_matrix):
    t = TfidfTransformer(norm='l2', smooth_idf=True, use_idf=True)
    tfidf_matrix = t.fit_transform(bow_matrix)
    return t, tfidf_matrix

In [67]:
# Extract BoW features
corpus = sample_df['cleaned_review'].tolist()
bow_vectorizer, bow_features = bow_extractor(corpus)


In [68]:
# get feature names
feature_names = bow_vectorizer.get_feature_names_out()

display_features(bow_features.todense(), feature_names)

      ab  abadoned  abandon  abandoned  abby  abedon  abel  abilities  \
0      0         0        0          0     0       0     0          0   
1      0         0        0          0     0       0     0          0   
2      0         0        0          0     0       0     0          0   
3      0         0        0          0     0       0     0          0   
4      0         0        0          0     0       0     0          0   
...   ..       ...      ...        ...   ...     ...   ...        ...   
4995   0         0        0          0     0       0     0          0   
4996   0         0        0          0     0       0     0          0   
4997   0         0        0          0     0       0     0          0   
4998   0         0        0          0     0       0     0          0   
4999   0         0        0          0     0       0     0          0   

      ability  able  ...  사양은  생각보다  없고요  에서는  윈도우로  이상  좋네요  타는편  편이네요  하면  
0           0     0  ...    0     0    0    0

In [69]:
# save BOW features to csv
save_features(bow_features.toarray(), feature_names, f'../text-mining/{n_gram_categ}_bow_features.csv')

✅ Features extracted and saved to ../text-mining/unigrams_bow_features.csv


In [71]:
# TFIDF extraction implementation using BOW features
tfidf_transform, tfidf_features = tfidf_transformer(bow_features)
features_idf = np.round(tfidf_features.todense(), 2)
display_features(features_idf, feature_names)

       ab  abadoned  abandon  abandoned  abby  abedon  abel  abilities  \
0     0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
1     0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
2     0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
3     0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
4     0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
...   ...       ...      ...        ...   ...     ...   ...        ...   
4995  0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
4996  0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
4997  0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
4998  0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   
4999  0.0       0.0      0.0        0.0   0.0     0.0   0.0        0.0   

      ability  able  ...  사양은  생각보다  없고요  에서는  윈도우로   이상  좋네요  타는편  편이네요   하면  
0         0.0   0.0  ...  0.0  

In [72]:
# save TFIDF features to csv
save_features(features_idf, feature_names, f'../text-mining/{n_gram_categ}_tfidf_features.csv')

✅ Features extracted and saved to ../text-mining/unigrams_tfidf_features.csv


# Text Classification using BOW